# B真实局部Wan读取：固定45帧E4（卡85）

当前已采用的用户任务：P0/PM各两片[60,105)、[61,106)，45帧全部进E，仅local latent0不读。
复用已存实际MP4 readback像素；VAEload1/E4/FFT4、两钥8读/8条件归约；D/codec/媒体decode/新writer/梯度0。
同一E供Q及FFT，两钥共享谱；L11和全部34起点不改。新272分数＋后评旧参考136＝408。
源cell身份、段号、局部raw bit、条件消息分开；32位每输入仍保留16存在/16结构缺失。

非对齐B的11名义组全mixed，精确source-cell准确率未定义，80单段兼容位/8跨段位未定义；不floor、dominant或消息救真值。
单查询不继承Card83跨窗碰撞拒读；错钥也可能有完整错误条件声称。身份/相位/FPR/可信消息未校准。
所有盲Q/分数/ties/原位票/条件消息及失败先存，之后才读取geometry/message/旧full-history参考后评。

直接Run all，从GitHub现有dev分支拉取源码后调用独立CLI，无需手工上传ZIP。Notebook不内嵌源码副本。
首个代码cell只Drive mount；attempt/科学残留存在则后续Run all仅显示保存结果，不补跑或覆盖源码。
无hash/manifest/版本/Git-clean/GPU型号或资源硬上限。agent仅CPU/static/fake与两固定几何检查，尚未真实执行本VAE/FFT任务。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, subprocess, tempfile
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Real-Local-Wan-V1/fixed-45-frame-phase01-v1')
def execution_artifacts(output):
    output=Path(output);names=('attempt.json','protocol.json','ledger.json','result.json','checkpoint.json','numeric_completion.json','evaluation.json')
    found=[n+s for n in names for s in ('','.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for folder in ('blind','features','preparation','references'):
        root=output/folder
        if root.is_dir():found.extend(str(p.relative_to(output)) for p in root.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)
SOURCE_REPOSITORY='https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH='dev/local-joint-state-payload-v1'
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path(tempfile.mkdtemp(prefix='b_real_local_wan_source_', dir='/content'))
    subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch',
                    SOURCE_BRANCH, SOURCE_REPOSITORY, str(SOURCE_ROOT)], check=True)
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('Available source', SOURCE_REPOSITORY, SOURCE_BRANCH, SOURCE_ROOT)
else:
    print('Existing attempt: saved records only; source/model not reentered')


沿已成功Wan依赖／缓存路径执行固定任务，只修复实际缺失导入。只读已存RGB像素，无FFmpeg安装或解码回退。

In [ ]:

import subprocess, sys

EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/"setup.log"
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode

    # Copied from the already-used A/P2 setup; no A import/build dependency.
    def ensure_imports(probe, packages, label):
        import re
        module_packages = {package.replace('-', '_'): package for package in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], ('Recheck ' if repaired else 'Probe ')+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named ['\"]([^'\"]+)['\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1) in module_packages:
                package = module_packages[missing.group(1)]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' failed without a known targeted repair; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, 'Repair '+label+': '+action)

    ensure_imports('import numpy,torch', ['numpy', 'torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
                   ['diffusers', 'accelerate', 'safetensors', 'huggingface-hub'], 'Wan VAE imports')
    with (OUTPUT/"runner.log").open("w") as log:
        process = subprocess.Popen([sys.executable, "-u", "-m",
            "experiments.wan_state_clock.b_real_local_wan_v1_run"],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    import time
    try:
        while process.poll() is None:
            time.sleep(30)
            ledger_path = OUTPUT/"ledger.json"
            if ledger_path.exists():
                ledger = json.loads(ledger_path.read_text())
                returned = sum(row["status"] == "RETURNED" for row in ledger.values())
                active = [name for name, row in ledger.items() if row["status"] == "STARTED"]
                print(f"Completed core units: {returned}/{len(ledger)}; active: {active}")
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Stopped; saved STARTED calls remain unresolved and will not be replayed.")
        raise
    print("Runner return code:", process.returncode)
else:
    print("Existing attempt: displaying saved records; no physical calls repeated.")
print("Output:", OUTPUT)


保存结果：技术失败、mixed/undefined、结构缺失、错误归属与假覆盖分别保留，不能由最高分认证。

In [ ]:

saved={}
for name in ('result.json','checkpoint.json','ledger.json','numeric_completion.json','evaluation.json'):
    path=OUTPUT/name
    if path.is_file():
        try:saved[name]=json.loads(path.read_text())
        except (OSError,ValueError) as exc:print('Unreadable saved record',name,type(exc).__name__,str(exc))
r=saved.get('result.json',saved.get('checkpoint.json',{}));e=saved.get('evaluation.json',{})
print('Execution',r.get('status'),'Fixed denominator',r.get('fixed_denominator'),'Calls',r.get('counts'))
for name,row in e.get('tables',{}).items():
    sequence=row.get('sequence',{})
    print(name,'arm',row.get('arm'),'singleton/mixed rows',row.get('singleton_geometry_rows'),row.get('mixed_identity_rows'),
          'provisional start',sequence.get('provisional_start'),'exact ties',sequence.get('exact_max_ties'),
          'conditional singleton correct/wrong',row.get('conditional_correct_singleton_cells'),row.get('conditional_wrong_singleton_cells'),
          'mixed unsupported claims',row.get('unsupported_mixed_singleton_claims'),
          'present readable/errors',row.get('readable_present_bits'),row.get('present_bit_errors'),
          'missing-group false coverage',row.get('conditional_false_missing_group_bits'),'undefined local targets',row.get('undefined_bit_references'))
print('Identity/phase',e.get('identity_status'),e.get('phase_status'),'trusted full message',e.get('trusted_full_messages'))
for name,row in saved.get('ledger.json',{}).items():
    if row.get('status')!='RETURNED':print('Retained failed/interrupted/unentered slot',name,row)
print('Full observations/votes: blind/. Exact geometry and all bit/source rows: evaluation.json.')
print('No floor truth for mixed cells; no identity/phase certification; no new writer/D/codec. Output',OUTPUT)
